In [10]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [3]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.6 MB/s eta 0:00:00


In [1]:
! pip install -q condacolab

In [2]:
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


In [3]:
! conda install -c bioconda seqkit

Channels:
 - bioconda
 - conda-forge
Platform: linux-64
Solving environment: - \ | / done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - seqkit


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.2.25  |       hbd8a1cb_0         144 KB  conda-forge
    certifi-2026.2.25          |     pyhd8ed1ab_0         148 KB  conda-forge
    conda-24.11.3              |  py311h38be061_0         1.1 MB  conda-forge
    openssl-3.6.1              |       h35e630c_1         3.0 MB  conda-forge
    seqkit-2.13.0              |       he881be0_0         6.6 MB  bioconda
    ------------------------------------------------------------
                                           Total:        11.0 MB

The following NEW packages will be INSTALLED:

  seqkit             bioconda/linux-64::seqkit-2.13.0-he881be0_0 

The following packages will b

In [4]:
import requests
import re
import json
from Bio import SeqIO
import subprocess
import sys

class MyFastaParser:
    def __init__(self, file_name):
        self.filename = file_name

    def _get_uniprot(self, accession):
        # From HW2
        url = f"https://rest.uniprot.org/uniprotkb/{accession}"
        params = {'format': 'json'}
        return requests.get(url, params=params)

    def _get_ensembl(self, id):
        # From HW2
        url = f"https://rest.ensembl.org/lookup/id/{id}"
        params = {"content-type": "application/json"}
        return requests.get(url, params=params)

    def _uniprot_parse_response(self, resp):
        # Adapted from HW2 to return just the inner dictionary for HW4 output format
        if not resp.ok:
            return None
        data = resp.json()
        try:
            return {
                'organism': data.get('organism', {}).get('scientificName'),
                'geneInfo': data.get('genes', []),
                'sequenceInfo': data.get('sequence', {}),
                'type': 'protein'
            }
        except Exception:
            return None

    def _ensembl_parse_response(self, resp):
        # Adapted from HW2 to return just the inner dictionary for HW4 output format
        if not resp.ok:
            return None
        data = resp.json()
        keys = [
            'object_type', 'assembly_name', 'species', 'db_type', 'biotype',
            'display_name', 'id', 'description', 'canonical_transcript', 'source'
        ]
        return {key: data.get(key) for key in keys}

    def _access_database(self, id, database, seq_description, seq_sequence) -> dict:
        # Generate the structured output required by HW4
        output = {
            f'file_info_{id}': {
                'description': seq_description,
                'sequence': seq_sequence
            }
        }

        db_data = None
        if database == 'uniprot':
            resp = self._get_uniprot(id)
            db_data = self._uniprot_parse_response(resp)
        elif database == 'ensembl':
            resp = self._get_ensembl(id)
            db_data = self._ensembl_parse_response(resp)

        if db_data:
            output[f'database_info_{id}'] = db_data
        else:
            output[f'database_info_{id}'] = {"error": f"Failed to retrieve data from {database}"}

        return output

    def seqkit_stats(self) -> dict:
        # Call seqkit via subprocess [cite: 33, 34]
        # We use '-a' to get all stats (Q1, Q2, N50, etc.) as shown in test_data.txt
        try:
            result = subprocess.run(
                ['seqkit', 'stats', '-a', '-T', self.filename],
                capture_output=True,
                text=True
            )

            if result.returncode != 0:
                return {'error': result.stderr.strip()}

            lines = result.stdout.strip().split('\n')
            if len(lines) < 2:
                return {'error': 'Unexpected seqkit output format'}

            # 严格按照 \t (Tab) 进行分割
            headers = lines[0].split('\t')
            values = lines[1].split('\t')

            stat_info = {}
            fasta_type = None
            fasta_num_seqs = None

            for h, v in zip(headers, values):
                if h == 'file':
                    continue
                stat_info[h] = v
                if h == 'type':
                    fasta_type = v
                if h == 'num_seqs':
                    fasta_num_seqs = int(v.replace(',', ''))

            return {
                'fasta_seqkit_stat_info': stat_info,
                'fasta_type': fasta_type,
                'fasta_num_seqs': fasta_num_seqs
            }

        except FileNotFoundError:
            return {'error': 'seqkit command not found. Please ensure it is installed and in PATH.'}
        except Exception as e:
            return {'error': str(e)}

    def biopython_parser(self, seqkit_result) -> dict:
        # Exit early if seqkit encountered an error
        if 'error' in seqkit_result:
            return seqkit_result

        # Check file type to determine database and regex pattern [cite: 35, 38]
        fasta_type = seqkit_result.get('fasta_type', '')
        if fasta_type == 'Protein':
            database = 'uniprot'
            pattern = r"[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}"
        elif fasta_type in ('DNA', 'RNA'):
            database = 'ensembl'
            pattern = r"ENS[A-Z]*[GTP][0-9]{11}"
        else:
            return {"error": f"Unknown or unsupported fasta type: {fasta_type}"}

        output = {'DB_name': database}
        warnings = set()

        # Iterate over sequences to read the descriptions [cite: 37]
        try:
            for record in SeqIO.parse(self.filename, 'fasta'): # [cite: 36]
                # Check each description for a possible ID
                match = re.search(pattern, record.description)

                if match:
                    seq_id = match.group(0)
                    # Perform an API call [cite: 39]
                    db_info = self._access_database(
                        id=seq_id,
                        database=database,
                        seq_description=record.description,
                        seq_sequence=str(record.seq)
                    )
                    output.update(db_info) # [cite: 40]
                else:
                    warnings.add('No ID match found.')

            if warnings:
                output['WARNING'] = warnings

        except Exception as e:
             return {"error": f"Biopython parsing error: {str(e)}"}

        return output

    def show_output(self, output, indent=0):
        for key, value in output.items():
            print('\t' * indent + str(key))
            # Handle sets (like the WARNING set) so they print correctly
            if isinstance(value, set):
                print('\t' * (indent + 1) + str(value))
            elif isinstance(value, dict):
                self.show_output(value, indent + 1)
            else:
                print('\t' * (indent + 1) + str(value))

In [5]:
parser = MyFastaParser('/content/drive/MyDrive/Colab Notebooks/SP/test_file.fasta')

stats = parser.seqkit_stats()
print("--- SeqKit Stats ---")
print(stats)
print("\n")

biopython_result = parser.biopython_parser(stats)

print("--- Parser Output ---")
parser.show_output(biopython_result)

--- SeqKit Stats ---
{'fasta_seqkit_stat_info': {'format': 'FASTA', 'type': 'Protein', 'num_seqs': '2', 'sum_len': '456', 'min_len': '29', 'avg_len': '228.0', 'max_len': '427', 'Q1': '29', 'Q2': '228', 'Q3': '427', 'sum_gap': '0', 'N50': '427', 'N50_num': '1', 'Q20(%)': '0', 'Q30(%)': '0', 'AvgQual': '0.00', 'GC(%)': '0.00', 'sum_n': '0'}, 'fasta_type': 'Protein', 'fasta_num_seqs': 2}


--- Parser Output ---
DB_name
	uniprot
file_info_P11473
	description
		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
	sequence
		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
database_info_P114

In [6]:
# 实例化解析器并传入你的测试文件
parser = MyFastaParser('/content/drive/MyDrive/Colab Notebooks/SP/uniprot_download.fasta')

# 1. 运行 seqkit stats
stats = parser.seqkit_stats()
print("--- SeqKit Stats ---")
print(stats)
print("\n")

biopython_result = parser.biopython_parser(stats)

print("--- Parser Output ---")
parser.show_output(biopython_result)

--- SeqKit Stats ---
{'fasta_seqkit_stat_info': {'format': 'FASTA', 'type': 'Protein', 'num_seqs': '7', 'sum_len': '3861', 'min_len': '180', 'avg_len': '551.6', 'max_len': '1382', 'Q1': '429', 'Q2': '441', 'Q3': '500', 'sum_gap': '0', 'N50': '468', 'N50_num': '3', 'Q20(%)': '0', 'Q30(%)': '0', 'AvgQual': '0.00', 'GC(%)': '0.00', 'sum_n': '0'}, 'fasta_type': 'Protein', 'fasta_num_seqs': 7}


--- Parser Output ---
DB_name
	uniprot
file_info_Q9R1A7
	description
		sp|Q9R1A7|NR1I2_RAT Nuclear receptor subfamily 1 group I member 2 OS=Rattus norvegicus OX=10116 GN=Nr1i2 PE=2 SV=1
	sequence
		MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLLPHLADVSTYMFKGVINFAKVISHFRELPIEDQISLLKGATFEMCILRFNTMFDTETGTWECGRLAYCFEDPNGGFQKLLLDPLMKFHCMLKKLQLREEEYVLMQAISLFSPDRPGVVQRSVVDQLQERFALTLKAYIECSRPYPAHRFLFLKIMAVLTELRSINAQQTQQL

In [7]:
parser = MyFastaParser('/content/drive/MyDrive/Colab Notebooks/SP/ensembl_download_1.fasta')

stats = parser.seqkit_stats()
print("--- SeqKit Stats ---")
print(stats)
print("\n")

biopython_result = parser.biopython_parser(stats)

print("--- Parser Output ---")
parser.show_output(biopython_result)

--- SeqKit Stats ---
{'fasta_seqkit_stat_info': {'format': 'FASTA', 'type': 'DNA', 'num_seqs': '6', 'sum_len': '86', 'min_len': '9', 'avg_len': '14.3', 'max_len': '23', 'Q1': '10', 'Q2': '14', 'Q3': '17', 'sum_gap': '0', 'N50': '16', 'N50_num': '3', 'Q20(%)': '0', 'Q30(%)': '0', 'AvgQual': '0.00', 'GC(%)': '45.35', 'sum_n': '0'}, 'fasta_type': 'DNA', 'fasta_num_seqs': 6}


--- Parser Output ---
DB_name
	ensembl
file_info_ENSMUST00000196221
	description
		ENSMUST00000196221.2 cds chromosome:GRCm39:14:54350925:54350933:1 gene:ENSMUSG00000096749.3 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd1 description:T cell receptor delta diversity 1 [Source:MGI Symbol;Acc:MGI:4439547]
	sequence
		ATGGCATAT
database_info_ENSMUST00000196221
	object_type
		Transcript
	assembly_name
		GRCm39
	species
		mus_musculus
	db_type
		core
	biotype
		TR_D_gene
	display_name
		Trdd1-202
	id
		ENSMUST00000196221
	description
		None
	canonical_transcript
		None
	source
		havana
file_info_ENSM

In [11]:
parser = MyFastaParser('/content/drive/MyDrive/Colab Notebooks/SP/ensembl_download_2.fasta')

stats = parser.seqkit_stats()
print("--- SeqKit Stats ---")
print(stats)
print("\n")

biopython_result = parser.biopython_parser(stats)

print("--- Parser Output ---")
parser.show_output(biopython_result)

--- SeqKit Stats ---
{'error': 'Unexpected seqkit output format'}


--- Parser Output ---
error
	Unexpected seqkit output format
